# FlowMatch Injection — Satellite-Footprint Guided Multi-View SLAT

**Research prototype.** Specializes the MV-SAM3D / SAM 3D Objects reconstruction
stack for **buildings** reconstructed from a mix of **ground-level photos** and
**satellite / near-nadir imagery**.

## The idea (your hypothesis, restated precisely)

MV-SAM3D reconstructs a 3D object by running a **two-stage flow-matching**
generator and fusing several views with **multi-diffusion** (each view denoises
the same shared latent; per-step velocities are averaged/weighted and integrated
together).

Problem: **DA3 estimates bad camera extrinsics for satellite views** (oblique,
far, weird scale). Feeding the *raw satellite image features* into the
multi-diffusion therefore injects noise, not signal.

Proposed fix, implemented here:

1. **Ground views** drive the multi-diffusion **normally** — their DINOv2 image
   features condition the denoiser exactly as in stock MV-SAM3D.
2. **Satellite views** are **NOT** used as ordinary views. Instead we extract
   the **building footprint / perimeter** (from the satellite mask and/or
   **OpenStreetMap**) and inject *that geometry* into the flow-matching ODE as a
   **guidance term** on the velocity — a "FlowMatch Injection".

So: *ground feature vectors do the appearance/geometry denoising; the satellite
contributes only a top-down footprint prior that steers the structure stage.*

## Where the injection happens (grounded in the actual code)

The footprint is a **top-down 2D cross-section**. The natural place to use it is
**Stage 1 (sparse structure)**, which decides *where voxels exist* (occupancy).
Stage 2 (SLAT) only paints per-voxel latent features on the already-chosen
voxels, so the footprint does not belong there.

```
              ground views ──► DINOv2 features ──► per-view velocity
                                                       │  (multi-diffusion fuse)
                                                       ▼
   satellite ─► footprint ─► 3D occupancy prior ─► +guidance velocity
   (mask/OSM)   (perimeter)   (extruded)            │
                                                    ▼
                                  Stage-1 sparse-structure flow ODE
                                                    │
                                                    ▼  coords (occupied voxels)
                                          Stage-2 SLAT flow (ground only)
                                                    ▼
                                          decode ► mesh / gaussians
```

> Everything below is heavily commented because this is a scaffold you will keep
> iterating on. Sections marked **CALIBRATE** contain coordinate-frame
> assumptions you should verify on your data.


## 0 — How the base pipeline actually works (reference)

A condensed map of the code this notebook hooks into. File paths are inside
`MV-SAM3D/sam3d_objects/`.

| Concept | Where | Note |
|---|---|---|
| **SLAT** (sparse structured latent) | `model/.../sparse/basic.py` `SparseTensor` | occupied voxel coords + 8-D per-voxel feature |
| **Stage 1** sparse-structure flow | `pipeline/inference_pipeline.py:1009` `sample_sparse_structure_multi_view` | dense `4096×8` shape tokens → `ss_decoder` → occupancy → `coords` |
| **Stage 2** SLAT flow | `inference_pipeline.py:1283` `sample_slat_multi_view_weighted` | `N_voxels×8` features at Stage-1 coords |
| **Flow ODE** | `model/backbone/generator/flow_matching/solver.py` | Euler: `x_{t+1}=x_t+dt·v`, `t: 0→1` (noise→data) |
| **velocity** | `flow_matching/model.py:221` `_generate_dynamics` | predicts `v ≈ x1 − x0` |
| **multi-diffusion fuse (Stage 1)** | `pipeline/multi_view_utils.py` `inject_generator_multi_view` | runs denoiser per view, averages/weights velocities |
| **weighted fuse (Stage 2)** | `pipeline/multi_view_weighted.py:767` `weighted_fusion_sparse` | per-latent weighted sum |
| **Stage-1 weights** | `multi_view_weighted.py:236` `compute_ss_entropy_weights` | **confidence only** (attention entropy) |
| **Stage-2 weights** | `utils/latent_weighting.py` `LatentWeightManager` | **entropy + visibility (+mixed)** |
| **"physics" injection** | `model/.../classifier_free_guidance.py` `PointmapCFG` | 3-way CFG using DA3 pointmap (Stage 1 only) |
| **image conditioning** | `model/.../dit/embedder/dino.py` (DINOv2) + `embedder_fuser.py` | per-view tokens → cross-attention |
| **per-view condition build** | `inference_pipeline.py:964` `get_multi_view_condition_input` | stacks `(num_views,B,L,D)` |
| **entry** | `MV-SAM3D/run_inference_weighted.py` `run_weighted_inference` | loads images/masks + `da3_output.npz` → `run_multi_view` |

### Why our monkey-patch composes cleanly

`sample_sparse_structure_multi_view` imports the fuser **locally at call time**:

```
# inference_pipeline.py:1035  (inside the method)
from sam3d_objects.pipeline.multi_view_utils import inject_generator_multi_view
```

A `from X import Y` *inside a function* re-reads `X.Y` every call. So if we
overwrite `multi_view_utils.inject_generator_multi_view` with a wrapper, the
Stage-1 sampler will use **our** version — no edits to the library source. Our
wrapper enters the original context manager (which installs the multi-diffusion
fused dynamics) and then composes a **footprint-guidance term** on top of the
fused velocity.


## 1 — Environment & install (self-contained, fresh Colab A100-80GB)

This section is **fully self-contained** — run it top to bottom in a brand-new
Colab session. It is the exact, known-working install sequence from
`mvsam3d_example_current.ipynb`.

**Before you start:**
1. **Runtime → Change runtime type → A100 GPU** (80 GB). The SLAT backbone + DA3
   on building imagery need the headroom.
2. **Add your Hugging Face token to Colab Secrets** as `HF_TOKEN` (sidebar key
   icon, toggle notebook access). `facebook/sam-3d-objects` is a **gated** repo —
   request access on its HF page first, or checkpoint download will 401.

**Restart gotcha (important):** Cell *1.2 (Step 0)* pins `torch==2.5.1+cu121`. On
a fresh runtime Colab ships a newer torch, so this cell will downgrade it and
print *"Restart the runtime now!"*. When you see that: **Runtime → Restart
session**, then re-run **from the clone cell (1.3) downward** (skip re-running
Step 0 — torch is already correct, and it must not be re-pinned).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import torch, subprocess, sys, os

assert torch.cuda.is_available(), "No GPU — enable a GPU runtime first!"
gpu_name = torch.cuda.get_device_name(0)
vram_gb  = torch.cuda.get_device_properties(0).total_memory / 1e9

print(f"GPU:     {gpu_name}")
print(f"VRAM:    {vram_gb:.1f} GB")
print(f"PyTorch: {torch.__version__}")
print(f"CUDA:    {torch.version.cuda}")
print(f"Python:  {sys.version}")

if vram_gb < 30:
    print("\n⚠  Less than 32 GB VRAM — inference may OOM on large batches. "
          "Consider switching to an A100 runtime.")

# SAM 3D Objects targets torch 2.5.1 + CUDA 12.1.
# Colab's default torch is often too new for pre-built wheels of
# flash-attn, kaolin, spconv, etc.  Pin the correct stack now.
TARGET_TORCH = "2.5.1"
if not torch.__version__.startswith(TARGET_TORCH):
    print(f"\n▶ Downgrading torch {torch.__version__} → {TARGET_TORCH}+cu121 …")
    !pip install -q \
        torch==2.5.1 torchvision==0.20.1 torchaudio==2.5.1 \
        --index-url https://download.pytorch.org/whl/cu121
    !pip install -q xformers==0.0.28.post3
    print("✓ Torch stack pinned.  ⚠ Restart the runtime now!")
    print("   Go to Runtime → Restart session, then re-run from this cell.")
else:
    print(f"\n✓ torch {torch.__version__} is the target version")

> **If Step 0 above printed "Restart the runtime now!":** do
> **Runtime → Restart session**, then continue from this cell (1.3). Do **not**
> re-run Step 0 after the restart. If it printed *"torch 2.5.1 is the target
> version"*, just keep going — no restart needed.

In [ ]:
# 1.1 — Define paths and clone the three repos (same remotes as the working notebook)
from pathlib import Path
import os, sys

ROOT         = Path("/content")
SAM3D_ROOT   = ROOT / "sam-3d-objects"
DA3_ROOT     = ROOT / "Depth-Anything-3"
MVSAM3D_ROOT = ROOT / "MV-SAM3D"

# Pin numpy AND scipy for EVERY pip install in this session. SAM3D + DA3 require
# numpy<2, but kaolin/scipy/etc. try to pull numpy 2.x — and upgrading numpy while it
# is already imported corrupts the live kernel ("No module named 'numpy.char' / fails
# sanity checks"). PIP_CONSTRAINT makes pip refuse to change them.
# scipy is pinned too: modern Colab ships a numpy-2-only scipy whose array-API shim
# does `from numpy import strings` (numpy>=2 only), so under numpy 1.26 `import scipy`
# explodes with "No module named 'numpy.strings'" — which crashes `import kaolin`.
# scipy 1.13.1 supports numpy 1.26 and has no such code path.
# NOTE: env vars do NOT survive a runtime restart — this cell re-sets it, so always
# re-run it (it's the first cell after Step 0) before the install cells.
_np_pin = ROOT / "_numpy_pin.txt"
_np_pin.write_text("numpy==1.26.4\nscipy==1.13.1\n")
os.environ["PIP_CONSTRAINT"] = str(_np_pin)
print("PIP_CONSTRAINT ->", os.environ["PIP_CONSTRAINT"], "(numpy 1.26.4 + scipy 1.13.1 locked)")

# Force-align scipy with numpy 1.26.4 NOW, before anything imports scipy this session.
# (PIP_CONSTRAINT prevents future upgrades but won't downgrade the scipy Colab preinstalled.)
# Doing it here — on a fresh kernel, before any `import scipy` — avoids stale broken
# modules; otherwise kaolin's import later dies on "No module named 'numpy.strings'".
!pip install -q "numpy==1.26.4" "scipy==1.13.1"
import importlib, scipy, numpy
importlib.reload(scipy)
print(f"✓ numpy {numpy.__version__}  scipy {scipy.__version__}")

%cd /content
REPOS = {
    "sam-3d-objects":   "https://github.com/facebookresearch/sam-3d-objects.git",
    "Depth-Anything-3": "https://github.com/ByteDance-Seed/Depth-Anything-3.git",
    "MV-SAM3D":         "https://github.com/devinli123/MV-SAM3D.git",
}
for name, url in REPOS.items():
    if os.path.isdir(name):
        print(f"OK   {name}/ already exists")
    else:
        !git clone --depth 1 {url}
        print(f"cloned {name}")


In [ ]:
# xformers must match the installed torch+cuda version.
# Colab usually ships a compatible pair, but let's be explicit.
!pip install -q xformers

# Install DA3 as an editable package (pulls in its own deps)
!pip install -q -e /content/Depth-Anything-3

print("\n✓ Depth Anything 3 installed")

In [ ]:
import torch

TORCH_VERSION = torch.__version__.split("+")[0]   # e.g. "2.10.0"
CUDA_TAG      = torch.version.cuda.replace(".", "")  # e.g. "128"
print(f"Targeting torch {TORCH_VERSION}  CUDA {torch.version.cuda}")

# ── 4a: Install the sam3d_objects package (code only, no deps) ──
!pip install -q --no-deps -e /content/sam-3d-objects

# ── 4b: Runtime deps (open3d and utils3d conflict on numpy, install separately) ──
!pip install -q \
    hydra-core==1.3.2 \
    omegaconf \
    loguru \
    opencv-python-headless \
    trimesh \
    roma \
    seaborn \
    safetensors \
    tqdm \
    einops \
    astor \
    lightning \
    xatlas \
    pyvista \
    pymeshfix \
    igraph \
    "numpy<2" \
    "huggingface_hub[cli]" \
    gradio

# open3d and utils3d have conflicting numpy pins — install each alone
!pip install -q open3d
!pip install -q --no-deps "utils3d @ git+https://github.com/EasternJournalist/utils3d.git@3913c65d81e05e47b9f367250cf8c0f7462a0900"

# ── 4c: MoGe (pointmap geometry utilities used by sam3d_objects) ──
!pip install -q --no-deps "MoGe @ git+https://github.com/microsoft/MoGe.git@a8c37341bc0325ca99b9d57981cc3bb2bd3e255b"

# The long list of "sam3d-objects requires X" warnings above is expected —
# we intentionally skipped those with --no-deps. They are dev/training deps
# not needed for inference.

print("\n✓ Core SAM 3D deps installed")

In [ ]:
import torch, subprocess, sys, os

TORCH_VER = torch.__version__.split("+")[0]            # "2.5.1"
CUDA_VER  = "cu" + torch.version.cuda.replace(".", "") # "cu121"
PY_VER    = f"{sys.version_info.major}{sys.version_info.minor}"

print(f"torch {TORCH_VER}  CUDA {torch.version.cuda}  Python {PY_VER}")
assert TORCH_VER.startswith("2.5"), (
    f"Expected torch 2.5.x but got {TORCH_VER}. "
    "Re-run Step 0 to pin the correct version, then restart the runtime."
)

# ── Re-assert the numpy/scipy pin for THIS cell (kaolin is the usual offender) ──
# Every `!pip install` below must honor numpy<2. If this cell runs on a kernel where
# PIP_CONSTRAINT isn't set (e.g. cell 1.1 wasn't re-run after a restart), `pip install
# kaolin` pulls numpy 2.x as a dependency and overwrites the numpy already loaded in
# this kernel -> 'No module named numpy.char' / 'fails sanity checks'. Recreate it here.
from pathlib import Path as _Path
_pin = _Path(os.environ.get("PIP_CONSTRAINT") or "")
if not (_pin and _pin.exists()):
    _pin = _Path("/content/_numpy_pin.txt")
    _pin.write_text("numpy==1.26.4\nscipy==1.13.1\n")
os.environ["PIP_CONSTRAINT"] = str(_pin)
print("PIP_CONSTRAINT ->", os.environ["PIP_CONSTRAINT"], "(numpy 1.26.4 + scipy 1.13.1 locked)")

!pip install -q ninja packaging

# ── Safety check: scipy must be numpy-1.26 compatible before kaolin imports it ──
# (cell 1.1 already downgrades scipy to 1.13.1; this just verifies / repairs.)
import scipy as _scipy
if not _scipy.__version__.startswith(("1.10", "1.11", "1.12", "1.13")):
    print(f"scipy {_scipy.__version__} is too new for numpy 1.26 — fixing …")
    !pip install -q "scipy==1.13.1"
    print("scipy downgraded. If `import kaolin` below still fails, RESTART the runtime")
    print("and re-run from cell 1.1 (scipy was already imported in this kernel).")
else:
    print(f"✓ scipy {_scipy.__version__} (numpy-1.26 compatible)")

# ── PyTorch3D ──
try:
    import pytorch3d
    print(f"✓ pytorch3d {pytorch3d.__version__}")
except ImportError:
    print("Installing PyTorch3D …")
    ret = subprocess.run(
        f"pip install --no-index --no-cache-dir pytorch3d "
        f"-f https://dl.fbaipublicfiles.com/pytorch3d/packaging/wheels/"
        f"py{PY_VER}_{CUDA_VER}_pyt{TORCH_VER.replace('.','')}/download.html",
        shell=True,
    )
    if ret.returncode != 0:
        print("Pre-built wheel not found — building from source (5-10 min) …")
        !pip install "pytorch3d @ git+https://github.com/facebookresearch/pytorch3d.git"
    import pytorch3d
    print(f"✓ pytorch3d {pytorch3d.__version__}")

# ── flash-attn (pre-built wheel exists for torch 2.5.1 + cu121) ──
try:
    import flash_attn
    print(f"✓ flash_attn {flash_attn.__version__}")
except ImportError:
    print("Installing flash-attn …")
    !pip install -q flash-attn --no-build-isolation
    import flash_attn
    print(f"✓ flash_attn {flash_attn.__version__}")

# ── kaolin (NVIDIA publishes wheels indexed by torch+cuda) ──
os.environ["PIP_FIND_LINKS"] = (
    f"https://nvidia-kaolin.s3.us-east-2.amazonaws.com/"
    f"torch-{TORCH_VER}_{CUDA_VER}.html"
)
try:
    import kaolin
    print(f"✓ kaolin {kaolin.__version__}")
except ImportError:
    print("Installing kaolin …")
    # --constraint is passed explicitly (belt-and-suspenders with PIP_CONSTRAINT) so the
    # kaolin dependency resolver can NEVER pull numpy 2.x into this live kernel.
    !pip install -q kaolin --constraint "$PIP_CONSTRAINT"
    # REPAIR numpy after kaolin's deps. Even with the version pinned to 1.26.4, kaolin's
    # dependency install can leave a MIXED numpy on disk: 1.26 metadata (so a version
    # check passes) but a numpy.random C-extension rebuilt against numpy 2.x. Importing
    # it then dies with 'numpy.dtype size changed ... Expected 96 ... got 88'. A clean
    # --force-reinstall rewrites EVERY numpy binary to a consistent 1.26.4 before import.
    !pip install -q --force-reinstall --no-deps "numpy==1.26.4"
    try:
        import kaolin
        print(f"✓ kaolin {kaolin.__version__}")
    except (ImportError, ValueError) as _e:
        raise RuntimeError(
            f"kaolin import still failing after numpy repair ({type(_e).__name__}: {_e}).\n"
            "This kernel already loaded a broken numpy, so it cannot recover in place.\n"
            "FIX: Runtime > Restart session, then run cell 1.1 FIRST, then re-run this cell.\n"
            "(The disk is now repaired, so the next run will import kaolin cleanly.)") from _e

# ── gsplat (Gaussian splatting renderer) ──
try:
    import gsplat
    print(f"✓ gsplat {getattr(gsplat, '__version__', 'ok')}")
except ImportError:
    print("Installing gsplat …")
    !pip install -q --no-build-isolation \
        "gsplat @ git+https://github.com/nerfstudio-project/gsplat.git@0b4dddf04cb687367602c01196913cde6a743d70"
    import gsplat
    print(f"✓ gsplat installed")

# ── spconv (sparse convolution, used by the 3D backbone) ──
# Two failure modes we have to defeat here, in order:
#  (1) ABI mismatch: spconv-cu121==2.3.8 is compiled against cumm 0.7.x and pins
#      cumm-cu121<0.8.0. Letting pip pull the *latest* cumm (>=0.8.0) gives a
#      core_cc that fails with 'could not convert default argument workspace:
#      tv::Tensor ... type not registered yet'. Fix: pin BOTH in one resolve.
#  (2) Bogus JIT recompile: on Colab, cumm misdetects an 'editable' install and
#      tries to recompile core_cc at import. That compile fails because cumm
#      probes /usr/local/lib/python3.12/dist-packages/include (exists but empty)
#      as its header root instead of cumm/include, so it can't find
#      tensorview/*.h. Fix: CUMM_DISABLE_JIT=1 / SPCONV_DISABLE_JIT=1 force use
#      of the prebuilt core_cc shipped in the wheel — no compile at all.
# These env vars are read once at first import of cumm.constants, so they MUST be
# set before anything imports cumm/spconv (true on a fresh kernel).
os.environ["CUMM_DISABLE_JIT"]   = "1"
os.environ["SPCONV_DISABLE_JIT"] = "1"
# Probe spconv.pytorch (NOT bare `import spconv`) — bare import is lazy and never
# loads the C extension, so it can't detect a broken binary.
try:
    import spconv.pytorch  # noqa: F401 — this is what actually loads core_cc
    print("✓ spconv installed")
except Exception:
    print("Installing spconv (pinned pair: spconv-cu121==2.3.8 + cumm-cu121==0.7.11) …")
    !pip install -q --force-reinstall "spconv-cu121==2.3.8" "cumm-cu121==0.7.11"
    # Drop any half-loaded spconv/cumm modules so the freshly installed binary is
    # picked up; if core_cc was never loaded this lets us recover without a restart.
    import sys as _sys
    for _m in [k for k in list(_sys.modules) if k == "spconv" or k.startswith("spconv.") or k == "cumm" or k.startswith("cumm.")]:
        del _sys.modules[_m]
    try:
        import spconv.pytorch  # noqa: F401
        print("✓ spconv installed (matched pair, JIT disabled)")
    except Exception as _e:
        raise RuntimeError(
            f"spconv still failing after pinning spconv-cu121==2.3.8 + cumm-cu121==0.7.11 "
            f"({type(_e).__name__}: {_e}).\n"
            "This kernel already imported a broken/JIT cumm, so it can't recover in place.\n"
            "FIX: Runtime > Restart session, then run cell 1.1 FIRST, then re-run this cell.\n"
            "(The disk now has the correct matched pair + JIT disabled, so the next run imports cleanly.)") from _e

# Final numpy repair: spconv/cumm force-reinstalls above can also re-link numpy, so do a
# clean rewrite (not a no-op version pin) to guarantee consistent 1.26.4 binaries on disk.
!pip install -q --force-reinstall --no-deps "numpy==1.26.4"
print("\n✓ Compiled extensions done (numpy force-reinstalled to a clean 1.26.4)")

In [ ]:
import os

# Colab has no CONDA_PREFIX — point CUDA_HOME to the system CUDA toolkit.
# inference.py does: os.environ["CUDA_HOME"] = os.environ["CONDA_PREFIX"]
# We pre-set both so that line doesn't crash.
cuda_home = "/usr/local/cuda"
os.environ["CUDA_HOME"]    = cuda_home
os.environ["CONDA_PREFIX"] = cuda_home
os.environ["LIDRA_SKIP_INIT"] = "true"

# Run the Hydra patch shipped with sam-3d-objects
!python /content/sam-3d-objects/patching/hydra
print("✓ Hydra patched")

### 1.7 — Hugging Face auth + checkpoints

Logs in with the `HF_TOKEN` Colab secret, then downloads the **SAM 3D Objects**
checkpoint (gated) and the **Depth-Anything-3** model. The SAM 3D download lands
in `sam-3d-objects/checkpoints/hf/` and is symlinked into
`MV-SAM3D/checkpoints/hf` in Section 9 so `pipeline.yaml` resolves.

In [ ]:
from google.colab import output
output.enable_custom_widget_manager()

In [ ]:
from google.colab import userdata
from huggingface_hub import login

# Retrieve the token securely from Colab Secrets
hf_token = userdata.get('HF_TOKEN')

# Login programmatically, bypassing the broken widget
login(token=hf_token)

In [ ]:
from huggingface_hub import snapshot_download
from google.colab import userdata

hf_token = userdata.get('HF_TOKEN')

dl_path = snapshot_download(
    repo_id="facebook/sam-3d-objects",
    token=hf_token,
    local_dir=str(SAM3D_ROOT / "checkpoints" / "hf"),
)
print(f"✓ SAM 3D checkpoint saved to {dl_path}")
!ls -lh {SAM3D_ROOT / "checkpoints" / "hf"}

In [ ]:
from huggingface_hub import snapshot_download

DA3_MODEL_ID = "depth-anything/DA3NESTED-GIANT-LARGE"
da3_cache = snapshot_download(repo_id=DA3_MODEL_ID)
print(f"✓ DA3 model cached at {da3_cache}")

In [ ]:
import sys, os

os.environ.setdefault("CUDA_HOME",       "/usr/local/cuda")
os.environ.setdefault("CONDA_PREFIX",    "/usr/local/cuda")
os.environ["LIDRA_SKIP_INIT"] = "true"

# MV-SAM3D root (so `from sam3d_objects.…` resolves to MV-SAM3D's copy)
sys.path.insert(0, str(MVSAM3D_ROOT))
# MV-SAM3D notebook helpers (inference.py, load_images_and_masks.py, mesh_alignment.py)
sys.path.insert(0, str(MVSAM3D_ROOT / "notebook"))
# DA3 source directory
sys.path.insert(0, str(DA3_ROOT / "src"))

# Smoke tests
from depth_anything_3.api import DepthAnything3;    print("✓ depth_anything_3")
from inference import Inference;                     print("✓ SAM 3D Inference")
from load_images_and_masks import load_images_and_masks_from_path; print("✓ load_images_and_masks")
from sam3d_objects.utils.cross_attention_logger import CrossAttentionLogger; print("✓ CrossAttentionLogger")
from sam3d_objects.utils.latent_weighting import WeightingConfig; print("✓ LatentWeightManager")
from pytorch3d.transforms import Transform3d;       print("✓ pytorch3d")

print("\n✓ All imports OK")

In [ ]:
# 1.9 — Extra deps for footprint / OSM work (cv2 already comes from the SAM 3D stack)
!pip install -q shapely
# OpenStreetMap footprints: osmnx (+ geopandas) is heavy, so it is NOT installed here.
# If you set USE_OSM=True, footprint_from_osm() auto-installs it on first call. To
# install eagerly instead, uncomment the next line:
# !pip install -q osmnx
!pip install -q numpy==1.26.4   # re-pin numpy AFTER any installs above

import numpy as np, cv2
print("numpy", np.__version__, "| cv2", cv2.__version__)


## 2 — Configuration

Two **separate input folders** — one for ground-level photos, one for satellite /
near-nadir shots.

```
DATA_ROOT/
├── ground/
│   └── <your photos>      img_007.jpg, img_042.jpg, ...   (drop them straight in ground/)
└── satellite/
    └── masks/             sat0.png, ...        (footprint mask; or use OSM instead)
```

**File names / numbering don't matter.** Photos can be `0.jpg, 1.jpg, ...` or
arbitrary names with gaps (`img_007.jpg`, `DSC_1183.jpg`, ...). Everything is
sorted naturally and images are matched to masks by their **stem** (filename
without extension), so gaps and non-sequential numbers are fine — they just need
to stay consistent between an image and its mask.

You do **not** need to pre-build `ground/images/` or `ground/masks/`, and you do
**not** need to move/rename anything by hand. Section 2.5 copies your raw
`ground/` photos into `ground/images/` and runs SAM3 to create
`ground/<MASK_NAME>/<stem>.png`. The two folder names are set by `MASK_NAME`
(default `"masks"`, giving `ground/images/` + `ground/masks/`); the object SAM3
looks for is set separately by `TEXT_PROMPT` (default `"building"`).

The satellite folder is consumed **only** to extract a footprint; its images are
never fed to the denoiser. Optionally set `USE_OSM=True` with a lat/lon to fetch
the footprint polygon from OpenStreetMap instead of a satellite mask.

In [ ]:
# 2.0 — Paths and hyper-parameters  (EDIT the <-- marked lines)
from pathlib import Path

DATA_ROOT = Path("/content/drive/MyDrive/SatelliteFuse/datasets/va_split")   # <-- EDIT
GROUND_DIR    = DATA_ROOT / "ground"
SATELLITE_DIR = DATA_ROOT / "satellite"
MASK_NAME     = "masks"             # <-- subfolder name holding the masks (e.g. ground/masks/)
TEXT_PROMPT   = "building"          # <-- SAM3 text prompt used to find the object

OUTPUT_DIR = Path("/content/flowmatch_out");  OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ---- OpenStreetMap footprint (optional alternative to the satellite mask) ----
USE_OSM   = True                   # <-- set True to fetch footprint from OSM
OSM_LATLON = (40.0, -75.0)          # <-- EDIT building lat/lon if USE_OSM
OSM_RADIUS_M = 60                   #     search radius (meters)

# ---- Canonical-frame assumptions (CALIBRATE on your data) --------------------
# SAM3D builds the object in a canonical cube. We extrude the footprint along the
# object "up" axis. Which grid axis is up, and how footprint rows/cols map to the
# canonical X/Z plane, depends on the trained model's convention — verify with the
# Stage-1 occupancy visualization further down and flip these if it looks rotated.
UP_AXIS        = "y"     # one of "x","y","z" — canonical vertical axis
FOOTPRINT_FLIP_ROWS = False   # flip footprint vertically before extrusion
FOOTPRINT_FLIP_COLS = False   # flip footprint horizontally
FOOTPRINT_TRANSPOSE = False   # swap the two footprint axes
HEIGHT_FRAC    = (0.0, 1.0)   # vertical span of the extrusion within the cube

# ---- FlowMatch guidance strength --------------------------------------------
GUIDANCE_SCALE = 0.5     # 0 = off. Strength of the footprint velocity term.
GUIDANCE_TMAX  = 0.8     # only apply while t <= TMAX (structure-forming phase)
GUIDANCE_MODE  = "x_t"   # "x_t" (decode current latent) or "x1" (decode clean estimate)
GUIDANCE_ENERGY = "bce"  # "bce" or "l2" silhouette matching
SS_GRID        = 64      # ss_decoder output occupancy resolution (TRELLIS = 64)

print("Ground dir   :", GROUND_DIR)
print("Satellite dir:", SATELLITE_DIR)
print("Guidance     : scale=%.2f tmax=%.2f mode=%s energy=%s up=%s"
      % (GUIDANCE_SCALE, GUIDANCE_TMAX, GUIDANCE_MODE, GUIDANCE_ENERGY, UP_AXIS))


## 2.5 — Segment the ground views with SAM3

`run_multi_view` needs a **per-view building mask** for every ground image so it
can separate the building from its surroundings. If you only have raw photos,
this section creates those masks for you:

1. **Organize** — drop your ground photos straight into `ground/` (no subfolder
   required, any filenames, gaps OK). We copy them into `ground/images/`, the
   layout the loader and the DA3 step expect.
2. **(Optional) Interactive points** — click each view to tell SAM3 which
   instance to keep (**left-click = keep**, **right-click = exclude**). Skip it to
   just use the `TEXT_PROMPT` (`"building"`) text prompt.
3. **Segment** — SAM3 writes RGBA masks to `ground/<MASK_NAME>/<stem>.png`
   (default `ground/masks/`).

After this, Section 3's loader (`mask_prompt=MASK_NAME`) finds everything with no
further changes. Masks are matched to images **by stem**, so the original photo
names/numbers are preserved and gaps are irrelevant.

> Need a satellite mask too (and not using OSM)? Put the satellite image(s) in
> `satellite/images/` and run the segmenter once more with
> `images_dir=SATELLITE_DIR/"images"`, `output_dir=SATELLITE_DIR` to fill
> `satellite/<MASK_NAME>/`.

> SAM3's weights (~3.4 GB) stream from Hugging Face on first use — make sure the
> HF-login cell in Section 1 ran successfully.

In [ ]:
# 2.5.1 — Install SAM3 (segmentation) and fix its tokenizer asset path
!pip install -q --no-deps sam3
!pip install -q decord ftfy iopath
!pip install -q numpy==1.26.4          # keep numpy pinned (PIP_CONSTRAINT enforces this too)

# SAM3's text encoder loads a BPE vocab from dist-packages/assets/. Locate & link it.
import glob, os, urllib.request
_target = "/usr/local/lib/python3.12/dist-packages/assets"
os.makedirs(_target, exist_ok=True)
_dst = f"{_target}/bpe_simple_vocab_16e6.txt.gz"
if os.path.exists(_dst):
    print("BPE vocab already present.")
else:
    _hits = (glob.glob("/usr/local/lib/python3.12/dist-packages/sam3/**/bpe_simple*", recursive=True)
             or glob.glob("/usr/local/lib/python3.12/dist-packages/**/bpe_simple_vocab_16e6*", recursive=True))
    if _hits:
        os.symlink(_hits[0], _dst); print("Linked BPE vocab from", _hits[0])
    else:
        urllib.request.urlretrieve(
            "https://github.com/openai/CLIP/raw/main/clip/bpe_simple_vocab_16e6.txt.gz", _dst)
        print("Downloaded BPE vocab ->", _dst)

In [ ]:
# 2.5.2 — SAM3 segmenter (one shared, cached processor for picker + batch run)
import torch, numpy as np
from pathlib import Path
from PIL import Image
from loguru import logger

_sam3_model_cache = {}   # picker (_preview_mask) and the segmenter both read this

def _get_sam3_processor(confidence_threshold: float = 0.1):
    """Build (once) and cache a SAM3 image processor; weights stream from HF."""
    if "processor" not in _sam3_model_cache:
        from sam3.model_builder import build_sam3_image_model
        from sam3.model.sam3_image_processor import Sam3Processor
        print("Loading SAM3 (one-time; first run downloads ~3.4 GB)...")
        model = build_sam3_image_model(load_from_HF=True, checkpoint_path=None)
        _sam3_model_cache["processor"] = Sam3Processor(model, confidence_threshold=confidence_threshold)
        print("SAM3 ready.")
    return _sam3_model_cache["processor"]


class SAM3MultiObjectSegmenter:
    """Segment one object across many views; optional per-view point disambiguation."""

    def __init__(self, processor=None, confidence_threshold: float = 0.1):
        self.processor = processor or _get_sam3_processor(confidence_threshold)

    @staticmethod
    def _select_mask_by_points(masks, points, labels, image_size):
        w, h = image_size
        best_idx, best_score = 0, -float("inf")
        for mi in range(len(masks)):
            m = masks[mi]
            m_np = m.squeeze(0).cpu().numpy() if torch.is_tensor(m) else (m.squeeze(0) if m.ndim > 2 else m)
            if m_np.shape != (h, w):
                m_pil = Image.fromarray((m_np * 255).astype(np.uint8)).resize((w, h), Image.NEAREST)
                m_np = np.array(m_pil) / 255.0
            m_bool = m_np > 0.5
            score = 0
            for (px, py), lbl in zip(points, labels):
                inside = m_bool[max(0, min(int(py), h - 1)), max(0, min(int(px), w - 1))]
                score += 1 if (lbl == 1) == bool(inside) else -1
            if score > best_score:
                best_score, best_idx = score, mi
        return best_idx

    def segment_object_multiview(self, images_dir, object_name, text_prompt, output_dir,
                                 per_view_points=None, image_names=None):
        images_dir, output_dir = Path(images_dir), Path(output_dir)
        logger.info(f"[Segmenting: {object_name}]  prompt='{text_prompt}'")
        if per_view_points:
            logger.info(f"  Per-view point prompts for {len(per_view_points)} view(s)")

        def _key(p):
            try:    return (0, int(p.stem), p.stem)
            except ValueError: return (1, 0, p.stem)

        all_files = sorted(list(images_dir.glob("*.png")) + list(images_dir.glob("*.jpg")), key=_key)
        # Map stem -> index in the FULL list so picker point keys stay valid even when we
        # only re-segment a subset of views below.
        full_index = {p.stem: idx for idx, p in enumerate(all_files)}
        image_files = all_files
        if image_names is not None:   # restrict to requested stems (e.g. only missing masks)
            _want = set(image_names)
            image_files = [p for p in all_files if p.stem in _want]
        if not image_files:
            return {"success": False, "error": "No images found", "success_views": 0, "total_views": 0}
        logger.info(f"  Processing {len(image_files)} views...")

        mask_dir = output_dir / object_name
        mask_dir.mkdir(parents=True, exist_ok=True)
        success, failed = 0, []

        for img_path in image_files:
            i = full_index[img_path.stem]   # index into the picker's full-list keys
            try:
                image = Image.open(img_path).convert("RGB")
                state = self.processor.set_image(image)
                out = self.processor.set_text_prompt(state=state, prompt=text_prompt)
                masks, scores = out["masks"], out["scores"]
                if len(masks) == 0:
                    logger.warning(f"  View {i}: no mask (confidence too low)"); failed.append(i); continue

                if per_view_points and i in per_view_points:
                    vp = per_view_points[i]
                    best_idx = self._select_mask_by_points(masks, vp["points"], vp["labels"], image.size)
                else:
                    best_idx = scores.argmax().item()

                best_mask, best_score = masks[best_idx], scores[best_idx].item()
                m_np = best_mask.squeeze(0).cpu().numpy() if torch.is_tensor(best_mask) else best_mask.squeeze(0)
                if m_np.shape != (image.size[1], image.size[0]):
                    m_np = np.array(Image.fromarray((m_np * 255).astype(np.uint8)).resize(image.size, Image.NEAREST)) / 255.0

                image_np, mask_bool = np.array(image), m_np > 0.5
                rgba = np.zeros((image_np.shape[0], image_np.shape[1], 4), dtype=np.uint8)
                rgba[mask_bool, :3] = image_np[mask_bool]
                rgba[mask_bool, 3] = 255
                Image.fromarray(rgba, "RGBA").save(mask_dir / f"{img_path.stem}.png")

                area = float((rgba[:, :, 3] > 0).mean())
                logger.info(f"  View {i}: {img_path.stem}.png  area={area*100:.1f}%  score={best_score:.3f}")
                success += 1
            except Exception as e:
                logger.error(f"  View {i}: failed - {e}"); failed.append(i)

        logger.info(f"  Result: {success}/{len(image_files)} views segmented")
        if failed:
            logger.warning(f"  Failed views: {failed}")
        return {"success": success > 0, "object_name": object_name, "total_views": len(image_files),
                "success_views": success, "failed_views": failed, "mask_dir": mask_dir}

print("SAM3MultiObjectSegmenter ready.")

In [ ]:
# 2.5.3 — Put raw ground images into ground/images/ (just drop them in `ground/`)
import shutil
from pathlib import Path

IMAGES_DIR  = GROUND_DIR / "images"
IMAGES_DIR.mkdir(parents=True, exist_ok=True)
MASK_PROMPT = TEXT_PROMPT        # the interactive picker preview below reads this

_exts = ("*.png", "*.jpg", "*.jpeg", "*.JPG", "*.JPEG", "*.PNG", "*.webp")
_flat = []
for _e in _exts:
    _flat += [p for p in GROUND_DIR.glob(_e) if p.is_file()]   # top-level only (not images/)

_copied = 0
for p in sorted(_flat):
    dst = IMAGES_DIR / p.name
    if not dst.exists():
        shutil.copy2(p, dst)        # non-destructive; swap for shutil.move to avoid duplicates
        _copied += 1

_n = len(list(IMAGES_DIR.glob("*.png")) + list(IMAGES_DIR.glob("*.jpg")))
print(f"ground/images/ ready: copied {_copied} new file(s); {_n} image(s) total")
print("  ->", IMAGES_DIR)
assert _n > 0, f"No images found. Put your ground photos directly in {GROUND_DIR}"

In [ ]:
# 2.5.4 — (OPTIONAL) Interactive per-view point picker. Skip to use the text prompt only.
## Interactive Per-View Point Picker for SAM3 Segmentation
#
# All images loaded into a JS viewer. Navigate with Prev/Next.
#   LEFT CLICK = positive (green), RIGHT CLICK = negative (red)
# Click "Save All to Python" when done to store PER_VIEW_POINTS.
# Views with no points use text-only segmentation.

import base64, json
from pathlib import Path
from IPython.display import display, HTML
from google.colab import output
from PIL import Image as PILImage

def _natural_sort_key(p):
    try:
        return (0, int(p.stem), p.stem)
    except ValueError:
        return (1, 0, p.stem)

image_files = sorted(
    list(IMAGES_DIR.glob("*.png")) + list(IMAGES_DIR.glob("*.jpg")),
    key=_natural_sort_key
)
N_IMAGES = len(image_files)

# Thumbnail images for the JS viewer (resize to max 800px wide for speed)
MAX_W = 800
img_data = []
img_dims = []
for f in image_files:
    im = PILImage.open(f)
    orig_w, orig_h = im.size
    img_dims.append((orig_w, orig_h))
    if orig_w > MAX_W:
        ratio = MAX_W / orig_w
        im = im.resize((MAX_W, int(orig_h * ratio)), PILImage.LANCZOS)
    import io
    buf = io.BytesIO()
    im.save(buf, format="JPEG", quality=70)
    img_data.append(base64.b64encode(buf.getvalue()).decode())

print(f"Loaded {N_IMAGES} images into viewer")

PER_VIEW_POINTS = {}

# Build JSON arrays for JS
_names_json = json.dumps([f.name for f in image_files])
_dims_json = json.dumps(img_dims)
_imgs_json = json.dumps(img_data)

def _receive_all_points(data_json):
    global PER_VIEW_POINTS
    raw = json.loads(data_json)
    PER_VIEW_POINTS = {}
    for k, v in raw.items():
        idx = int(k)
        if v:
            PER_VIEW_POINTS[idx] = {
                "points": [[p["x"], p["y"]] for p in v],
                "labels": [p["l"] for p in v],
            }
    n = sum(1 for v in PER_VIEW_POINTS.values() if v["points"])
    print(f"Saved {n}/{N_IMAGES} views with point prompts")
    for k in sorted(PER_VIEW_POINTS):
        v = PER_VIEW_POINTS[k]
        s = ", ".join(f"{'+'if l==1 else '-'}({p[0]},{p[1]})" for p, l in zip(v["points"], v["labels"]))
        print(f"  [{k}] {image_files[k].name}: {s}")

_sam3_model_cache = {}

def _preview_mask(img_idx, points_json):
    """Run SAM3 on one image, score masks with points, return overlay as base64."""
    import io, numpy as np, torch
    from PIL import Image as PILImg

    img_idx = int(img_idx)
    pts_raw = json.loads(points_json)
    if not pts_raw:
        return ""

    pts = [[p["x"], p["y"]] for p in pts_raw]
    lbls = [p["l"] for p in pts_raw]

    proc = _get_sam3_processor()
    img = PILImg.open(image_files[img_idx]).convert("RGB")
    w, h = img.size

    state = proc.set_image(img)
    out = proc.set_text_prompt(state=state, prompt=MASK_PROMPT.split(",")[0].strip())
    masks = out["masks"]
    if len(masks) == 0:
        return ""

    best_idx, best_score = 0, -float("inf")
    for mi in range(len(masks)):
        m = masks[mi]
        m_np = m.squeeze(0).cpu().numpy() if torch.is_tensor(m) else m.squeeze(0)
        if m_np.shape != (h, w):
            m_pil = PILImg.fromarray((m_np * 255).astype(np.uint8))
            m_pil = m_pil.resize((w, h), PILImg.NEAREST)
            m_np = np.array(m_pil) / 255.0
        m_bool = m_np > 0.5
        score = 0
        for (px, py), lbl in zip(pts, lbls):
            inside = m_bool[min(int(py), h-1), min(int(px), w-1)]
            score += 1 if (lbl == 1) == inside else -1
        if score > best_score:
            best_score = score
            best_idx = mi

    best_m = masks[best_idx]
    m_np = best_m.squeeze(0).cpu().numpy() if torch.is_tensor(best_m) else best_m.squeeze(0)
    if m_np.shape != (h, w):
        m_pil = PILImg.fromarray((m_np * 255).astype(np.uint8))
        m_pil = m_pil.resize((w, h), PILImg.NEAREST)
        m_np = np.array(m_pil) / 255.0

    img_np = np.array(img)
    overlay = img_np.copy()
    mask_bool = m_np > 0.5
    overlay[mask_bool] = (overlay[mask_bool] * 0.5 + np.array([255, 80, 80]) * 0.5).astype(np.uint8)

    thumb = PILImg.fromarray(overlay)
    if w > MAX_W:
        ratio = MAX_W / w
        thumb = thumb.resize((MAX_W, int(h * ratio)), PILImg.LANCZOS)
    buf = io.BytesIO()
    thumb.save(buf, format="JPEG", quality=75)
    return base64.b64encode(buf.getvalue()).decode()

output.register_callback('_receive_all_points', _receive_all_points)
output.register_callback('_preview_mask', _preview_mask)

html = f"""
<style>
#vw{{position:relative;display:inline-block;cursor:crosshair}}
#vw img{{display:block;max-width:900px}}
.mk{{position:absolute;width:18px;height:18px;border-radius:50%;border:2px solid #fff;
     transform:translate(-50%,-50%);pointer-events:none;font-size:11px;text-align:center;
     line-height:18px;color:#fff;font-weight:bold}}
.mp{{background:rgba(0,200,0,.85)}}.mn{{background:rgba(220,0,0,.85)}}
#hd{{font-size:15px;margin-bottom:6px}}
#pl{{margin-top:6px;font-family:monospace;font-size:13px;min-height:20px}}
#ct{{margin-top:8px}}
#ct button{{padding:6px 18px;margin-right:8px;font-size:14px;cursor:pointer}}
.bg{{background:#4CAF50;color:#fff;border:none;border-radius:4px}}
.bo{{background:#2196F3;color:#fff;border:none;border-radius:4px}}
</style>
<div id="hd"></div>
<div id="vw"><img id="im" /></div>
<div id="pl"></div>
<div id="ct">
  <button onclick="prev()">&larr; Prev</button>
  <button onclick="undo()">Undo</button>
  <button onclick="clr()">Clear</button>
  <button onclick="next()">Next &rarr;</button>
  <button class="bo" onclick="previewMask()">Preview Mask</button>
  <button class="bg" onclick="saveAll()">Save All to Python</button>
</div>
<div id="pv" style="margin-top:6px;font-style:italic;color:#666"></div>

<script>
var imgs={_imgs_json};
var names={_names_json};
var dims={_dims_json};
var N=imgs.length;
var allPts={{}};
var ci=0;

function show(i){{
  ci=i;
  document.getElementById('im').src='data:image/jpeg;base64,'+imgs[i];
  document.getElementById('hd').innerHTML=
    '<b>View '+i+'/'+(N-1)+'</b> — '+names[i]+' ('+dims[i][0]+'x'+dims[i][1]+
    ') | Left=<span style="color:green">+pos</span> Right=<span style="color:red">-neg</span>';
  renderPts();
}}

function getC(e){{
  var r=document.getElementById('im').getBoundingClientRect();
  var ow=dims[ci][0],oh=dims[ci][1];
  return[Math.round((e.clientX-r.left)/r.width*ow),
         Math.round((e.clientY-r.top)/r.height*oh)];
}}

document.getElementById('vw').addEventListener('click',function(e){{
  e.preventDefault();var c=getC(e);
  if(!allPts[ci])allPts[ci]=[];
  allPts[ci].push({{x:c[0],y:c[1],l:1}});renderPts();
}});
document.getElementById('vw').addEventListener('contextmenu',function(e){{
  e.preventDefault();var c=getC(e);
  if(!allPts[ci])allPts[ci]=[];
  allPts[ci].push({{x:c[0],y:c[1],l:0}});renderPts();
}});

function renderPts(){{
  var co=document.getElementById('vw');
  co.querySelectorAll('.mk').forEach(m=>m.remove());
  var pts=allPts[ci]||[];
  var r=document.getElementById('im').getBoundingClientRect();
  var ow=dims[ci][0],oh=dims[ci][1];
  pts.forEach(function(p){{
    var m=document.createElement('div');
    m.className='mk '+(p.l===1?'mp':'mn');
    m.style.left=(p.x/ow*r.width)+'px';
    m.style.top=(p.y/oh*r.height)+'px';
    m.textContent=p.l===1?'+':'-';
    co.appendChild(m);
  }});
  var el=document.getElementById('pl');
  if(pts.length===0){{el.innerHTML='No points (text-only for this view)';}}
  else{{el.innerHTML=pts.map(p=>(p.l===1?'<span style="color:green">+</span>':'<span style="color:red">-</span>')+'('+p.x+','+p.y+')').join(' ');}}
  // show count of annotated views
  var n=Object.keys(allPts).filter(k=>allPts[k].length>0).length;
  document.getElementById('hd').innerHTML+=' | <b>'+n+' views annotated</b>';
}}

function undo(){{if(allPts[ci]&&allPts[ci].length)allPts[ci].pop();renderPts();}}
function clr(){{allPts[ci]=[];renderPts();}}
function prev(){{if(ci>0)show(ci-1);}}
function next(){{if(ci<N-1)show(ci+1);}}
function saveAll(){{
  google.colab.kernel.invokeFunction('_receive_all_points',[JSON.stringify(allPts)],{{}});
}}
var _previewing=false;
async function previewMask(){{
  var pts=allPts[ci]||[];
  if(pts.length===0){{document.getElementById('pv').innerHTML='Add points first';return;}}
  if(_previewing)return;
  _previewing=true;
  document.getElementById('pv').innerHTML='Running SAM3 preview...';
  try{{
    var res=await google.colab.kernel.invokeFunction('_preview_mask',[ci,JSON.stringify(pts)],{{}});
    var b64=res.data['text/plain'].replace(/^'/,'').replace(/'$/,'');
    if(b64){{
      document.getElementById('im').src='data:image/jpeg;base64,'+b64;
      document.getElementById('pv').innerHTML='Showing mask overlay. Click Prev/Next to return to original.';
    }} else {{
      document.getElementById('pv').innerHTML='No masks found for this view.';
    }}
  }}catch(e){{
    document.getElementById('pv').innerHTML='Preview error: '+e;
  }}
  _previewing=false;
}}

show(0);
</script>
"""
display(HTML(html))

In [ ]:
# 2.5.5 — Run SAM3 over the ground views -> ground/building/<stem>.png
# Only segments when masks are missing. Set FORCE_RESEGMENT=True to overwrite.
FORCE_RESEGMENT = False

MASK_DIR = GROUND_DIR / MASK_NAME

def _stems(d):
    if not d.exists():
        return set()
    return {p.stem for p in (list(d.glob("*.png")) + list(d.glob("*.jpg")))}

_image_stems = _stems(IMAGES_DIR)
_mask_stems  = _stems(MASK_DIR)
_missing     = sorted(_image_stems - _mask_stems)

if not FORCE_RESEGMENT and _missing == [] and _mask_stems:
    print(f"All {len(_mask_stems)} mask(s) already exist in {MASK_DIR} — skipping SAM3.")
    print("  (set FORCE_RESEGMENT=True above to regenerate them)")
    seg_result = {"success_views": len(_mask_stems), "total_views": len(_image_stems)}
else:
    if _mask_stems and not FORCE_RESEGMENT:
        print(f"{len(_mask_stems)} mask(s) exist; segmenting {len(_missing)} missing view(s): {_missing}")
        _names = _missing
    else:
        if FORCE_RESEGMENT:
            print("FORCE_RESEGMENT=True — (re)segmenting all views.")
        _names = None   # segment everything

    PER_VIEW_POINTS = globals().get("PER_VIEW_POINTS", {})
    _pts = {k: v for k, v in PER_VIEW_POINTS.items() if v.get("points")} if PER_VIEW_POINTS else {}
    print(f"Segmenting prompt {TEXT_PROMPT!r} -> {MASK_NAME}/"
          + (f" with point prompts for {len(_pts)} view(s)" if _pts else " (text prompt only)"))

    segmenter = SAM3MultiObjectSegmenter(confidence_threshold=0.1)
    seg_result = segmenter.segment_object_multiview(
        images_dir=IMAGES_DIR, object_name=MASK_NAME, text_prompt=TEXT_PROMPT,
        output_dir=GROUND_DIR, per_view_points=_pts or None, image_names=_names,
    )
    print(f"\n{seg_result['success_views']}/{seg_result['total_views']} ground masks -> {MASK_DIR}")

_total_masks = len(_stems(MASK_DIR))
assert _total_masks > 0, "No masks present — check the SAM3 prompt and your images."

## 3 — Load data

Ground images + masks are loaded with MV-SAM3D's own loader so the format matches
exactly what `run_multi_view` expects. Satellite images/masks are loaded plainly
(only the mask matters for the footprint).


In [ ]:
# 3.0 — Load ground views with MV-SAM3D's loader (RGB + mask, matched by stem)
from load_images_and_masks import load_images_and_masks_from_path

ground_images, ground_masks, ground_names = load_images_and_masks_from_path(
    GROUND_DIR, mask_prompt=MASK_NAME, image_names=None,
)
print(f"Ground views: {len(ground_images)}  names={ground_names}")

# Sanity: show shapes of the first view
import numpy as np
print("img0", np.array(ground_images[0]).shape, "mask0", np.array(ground_masks[0]).shape)


In [ ]:
# 3.1 — Load satellite mask(s).  We only need the building silhouette(s).
import numpy as np
from PIL import Image
import glob, os

def _load_mask(path):
    m = np.array(Image.open(path).convert("L"))
    return (m > 127).astype(np.uint8)

sat_masks = []
sat_mask_dir = SATELLITE_DIR / MASK_NAME
if sat_mask_dir.exists():
    for f in sorted(glob.glob(str(sat_mask_dir / "*.png")) + glob.glob(str(sat_mask_dir / "*.jpg"))):
        sat_masks.append(_load_mask(f))
    print(f"Loaded {len(sat_masks)} satellite mask(s) from {sat_mask_dir}")
else:
    print(f"No satellite mask dir at {sat_mask_dir} (that is fine if USE_OSM=True)")


## 4 — Extract the building footprint (perimeter)

A **footprint** is a normalized top-down occupancy image in `[0,1]` on a square
canvas (canonical XY). Two sources:

* **Satellite mask** → largest external contour → square-cropped, resized.
* **OSM** → building polygon near a lat/lon → rasterized.

If both are available you can intersect/union them; by default we take the
satellite mask when present, else OSM.


In [ ]:
# 4.0 — Footprint from a satellite building mask
import cv2, numpy as np

def footprint_from_mask(mask, out_size=256):
    """Largest building contour -> square, axis-aligned footprint in [0,1].

    Returns (footprint[out_size,out_size] float32, polygon Nx2 in canvas coords).
    """
    m = (np.asarray(mask) > 0).astype(np.uint8)
    contours, _ = cv2.findContours(m, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not contours:
        raise ValueError("No building contour found in satellite mask")
    c = max(contours, key=cv2.contourArea).reshape(-1, 2)

    x, y, w, h = cv2.boundingRect(c.astype(np.int32))
    side = max(w, h)
    cx, cy = x + w / 2.0, y + h / 2.0
    x0, y0 = cx - side / 2.0, cy - side / 2.0

    canvas = np.zeros((side, side), np.uint8)
    shifted = (c - np.array([x0, y0])).astype(np.int32)
    cv2.fillPoly(canvas, [shifted], 1)

    foot = cv2.resize(canvas, (out_size, out_size), interpolation=cv2.INTER_NEAREST)
    poly = shifted * (out_size / float(side))
    return foot.astype(np.float32), poly


In [ ]:
# 4.1 — (Optional) Footprint from OpenStreetMap
def footprint_from_osm(lat, lon, radius_m=60, out_size=256):
    """Fetch building polygons near (lat,lon), pick the one containing/closest to
    the point, rasterize to a normalized footprint. Requires `osmnx` + `shapely`."""
    try:
        import osmnx as ox
    except ModuleNotFoundError:
        print("osmnx not found — installing (one-time)…")
        import subprocess, sys
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "osmnx"], check=True)
        import osmnx as ox
    from shapely.geometry import Point
    gdf = ox.features_from_point((lat, lon), tags={"building": True}, dist=radius_m)
    gdf = gdf[gdf.geometry.type.isin(["Polygon", "MultiPolygon"])].copy()
    if len(gdf) == 0:
        raise ValueError("No OSM building polygons found near point")

    pt = Point(lon, lat)
    containing = gdf[gdf.contains(pt)]
    geom = (containing.iloc[0].geometry if len(containing)
            else gdf.iloc[gdf.distance(pt).values.argmin()].geometry)
    if geom.geom_type == "MultiPolygon":
        geom = max(geom.geoms, key=lambda g: g.area)

    xs, ys = geom.exterior.coords.xy
    xs, ys = np.array(xs), np.array(ys)   # lon, lat in degrees
    # Convert degrees -> approximate local meters so the footprint keeps its TRUE
    # aspect ratio. 1 deg latitude is ~constant, but 1 deg longitude shrinks by
    # cos(latitude); without this the shape gets stretched east-west away from the equator.
    import math
    mx = (xs - xs.min()) * math.cos(math.radians(float(ys.mean())))   # easting (rel. meters)
    my = (ys - ys.min())                                              # northing
    side = max(mx.max(), my.max()) + 1e-9
    u = mx / side * (out_size - 1)
    v = (out_size - 1) - (my / side * (out_size - 1))   # flip so North points up in the raster
    canvas = np.zeros((out_size, out_size), np.uint8)
    cv2.fillPoly(canvas, [np.stack([u, v], 1).astype(np.int32)], 1)
    return canvas.astype(np.float32), np.stack([u, v], 1)


In [ ]:
# 4.2 — Build THE footprint we will use
import matplotlib.pyplot as plt

if USE_OSM:
    footprint, poly = footprint_from_osm(*OSM_LATLON, radius_m=OSM_RADIUS_M, out_size=256)
    src = "OSM"
elif len(sat_masks) > 0:
    # If multiple satellite masks, use the one with the largest area (most nadir / closest)
    idx = int(np.argmax([m.sum() for m in sat_masks]))
    footprint, poly = footprint_from_mask(sat_masks[idx], out_size=256)
    src = f"satellite mask #{idx}"
else:
    raise RuntimeError("No footprint source: provide satellite masks or set USE_OSM=True")

print(f"Footprint from {src}: occupied fraction = {footprint.mean():.3f}")
plt.figure(figsize=(4,4)); plt.imshow(footprint, cmap="gray"); plt.title(f"footprint ({src})"); plt.axis("off"); plt.show()


## 5 — Footprint → canonical 3D occupancy prior

We extrude the 2D footprint along the canonical **UP_AXIS** to make a soft 3D
occupancy volume `prior ∈ [0,1]^{G×G×G}` (G = `SS_GRID`). The guidance compares
the **top-down silhouette** of the model's decoded occupancy against the
footprint (silhouette guidance — robust to unknown height/scale). The full 3D
prior is also built for optional volumetric experiments.

**CALIBRATE:** `UP_AXIS`, the flips and transpose set how the footprint maps onto
the canonical cube. Use the Stage-1 occupancy overlay later to fix orientation.


In [ ]:
# 5.0 — Orientation-adjust the footprint, then extrude to a 3D prior
import numpy as np, cv2

def orient_footprint(foot):
    f = foot.copy()
    if FOOTPRINT_TRANSPOSE: f = f.T
    if FOOTPRINT_FLIP_ROWS: f = f[::-1, :]
    if FOOTPRINT_FLIP_COLS: f = f[:, ::-1]
    return np.ascontiguousarray(f)

def footprint_to_prior(foot, grid=64, up_axis="y", height_frac=(0.0, 1.0)):
    """Extrude a 2D footprint into a (grid,grid,grid) occupancy volume."""
    f = cv2.resize(orient_footprint(foot), (grid, grid), interpolation=cv2.INTER_AREA)
    f = np.clip(f, 0.0, 1.0).astype(np.float32)
    prior = np.zeros((grid, grid, grid), np.float32)   # indices [x, y, z]
    k0, k1 = int(height_frac[0] * grid), int(height_frac[1] * grid)
    for k in range(k0, k1):
        if   up_axis == "y": prior[:, k, :] = f     # footprint lives in the X-Z plane
        elif up_axis == "z": prior[:, :, k] = f
        else:                prior[k, :, :] = f
    return prior, f

prior3d, foot_grid = footprint_to_prior(footprint, grid=SS_GRID, up_axis=UP_AXIS, height_frac=HEIGHT_FRAC)
print("prior3d", prior3d.shape, "occupied frac", prior3d.mean().round(4))

# Visualize the three orthographic projections of the prior
fig, ax = plt.subplots(1, 3, figsize=(10, 3.3))
ax[0].imshow(prior3d.max(0), cmap="viridis"); ax[0].set_title("proj along X")
ax[1].imshow(prior3d.max(1), cmap="viridis"); ax[1].set_title("proj along Y (top-down)")
ax[2].imshow(prior3d.max(2), cmap="viridis"); ax[2].set_title("proj along Z")
for a in ax: a.axis("off")
plt.tight_layout(); plt.show()


## 6 — The FlowMatch Injection

The Stage-1 dense shape latent `z` has shape `(B, 4096, 8)`. The pipeline decodes
it to an occupancy volume exactly like this (`inference_pipeline.py:1176`):

```
ss = ss_decoder( z.permute(0,2,1).contiguous().view(B, 8, 16, 16, 16) )   # -> (B,1,64,64,64) logits
```

**Energy.** Project the decoded occupancy to a top-down silhouette (max over the
up-axis) and compare to the footprint via BCE or L2:

```
E(z) = match( topdown( sigmoid(ss_decoder(z)) ),  footprint )
```

**Guidance velocity.** We add `−scale · ∇_z E` to the **fused** Stage-1 velocity.
Because the Euler step is `z ← z + dt·v` with `t: 0→1` (noise→data), pushing `v`
down the energy gradient nudges the forming structure toward the footprint.

* `GUIDANCE_MODE="x_t"` differentiates the energy at the current latent `z=x_t`
  (simple, convention-free, default).
* `GUIDANCE_MODE="x1"` first forms the clean-sample estimate
  `x1 ≈ x_t + (1−t)·v` (linear-FM relation) and differentiates there — sharper
  late in sampling but assumes the standard (non-reversed) schedule.

`GUIDANCE_TMAX` disables guidance once `t > TMAX` so we only shape the coarse
structure, leaving fine denoising to the ground views.


In [ ]:
# 6.0 — Footprint guidance module
import torch, torch.nn.functional as F

class FootprintGuidance:
    """Adds a footprint-silhouette guidance velocity to the Stage-1 flow."""

    def __init__(self, footprint_grid, ss_decoder, *, up_axis="y", scale=0.5,
                 t_max=0.8, mode="x_t", energy="bce", device="cuda"):
        # footprint_grid: 2D [G,G] top-down target in [0,1] (already oriented)
        self.target = torch.as_tensor(footprint_grid, dtype=torch.float32, device=device)
        self.ss_decoder = ss_decoder
        self.up_axis, self.scale, self.t_max = up_axis, float(scale), float(t_max)
        self.mode, self.energy_type, self.device = mode, energy, device
        self._logged = False

    def _topdown(self, occ):
        # occ: (B,1,G,G,G) probabilities -> (B,G,G) silhouette by max over up-axis
        o = occ[:, 0]
        if   self.up_axis == "y": sil = o.amax(dim=2)   # over Y -> (B, X, Z)
        elif self.up_axis == "z": sil = o.amax(dim=3)
        else:                     sil = o.amax(dim=1)
        return sil

    def _energy(self, shape_latent):
        B = shape_latent.shape[0]
        vol = shape_latent.permute(0, 2, 1).contiguous().view(B, 8, 16, 16, 16)
        logits = self.ss_decoder(vol)                      # (B,1,G,G,G)
        sil = self._topdown(torch.sigmoid(logits))         # (B,Gx,Gz)
        tgt = self.target
        if tgt.shape != sil.shape[-2:]:
            tgt = F.interpolate(tgt[None, None].float(), size=sil.shape[-2:], mode="bilinear",
                                align_corners=False)[0, 0]
        tgt = tgt.expand_as(sil)
        # Stage-1 runs under torch.autocast (fp16). binary_cross_entropy is NOT
        # autocast-safe, so disable autocast and evaluate the loss in fp32. The
        # decoder forward above stays under autocast (matches its param dtypes);
        # only the loss math is forced to fp32 here.
        with torch.autocast(device_type=sil.device.type, enabled=False):
            sil = sil.float()
            tgt = tgt.float()
            if self.energy_type == "bce":
                return F.binary_cross_entropy(sil.clamp(1e-4, 1 - 1e-4), tgt)
            return F.mse_loss(sil, tgt)

    def velocity(self, x_t, t, fused_v=None):
        """Return the guidance velocity (same shape as the shape latent)."""
        t_val = float(t.reshape(-1)[0]) if torch.is_tensor(t) else float(t)
        if self.scale == 0.0 or t_val > self.t_max:
            return torch.zeros_like(x_t)
        z = x_t
        if self.mode == "x1" and fused_v is not None:
            z = x_t + (1.0 - t_val) * fused_v        # clean-sample estimate (linear FM)
        with torch.enable_grad():
            zin = z.detach().float().requires_grad_(True)
            E = self._energy(zin)
            (grad,) = torch.autograd.grad(E, zin)
        if not self._logged:
            print(f"[FootprintGuidance] t={t_val:.3f} E={float(E):.4f} "
                  f"|grad|={float(grad.norm()):.4f} scale={self.scale}")
            self._logged = True
        return (-self.scale * grad).to(x_t.dtype)


In [ ]:
# 6.1 — Compose guidance with the fused multi-diffusion dynamics
def make_guided_dynamics(fused_dynamics, guidance):
    """Wrap a fused dynamics fn so it adds footprint guidance to the SHAPE velocity."""
    # The footprint guidance only makes sense for the STAGE-1 sparse-structure latent
    # (B, 4096, 8) -> (B, 8, 16, 16, 16). Stage-2 reuses multidiffusion with a 'shape'
    # key too, but that is the structured latent (B, N_voxels, 8) with a variable voxel
    # count, which must NOT be fed to the SS decoder. Gate on the 4096-token signature.
    SS_TOKENS = 4096    # 16^3 sparse-structure tokens
    def _is_ss_shape(t):
        return torch.is_tensor(t) and t.dim() == 3 and t.shape[1] == SS_TOKENS and t.shape[2] == 8
    def guided(x_t, t, *args, **kwargs):
        v = fused_dynamics(x_t, t, *args, **kwargs)
        # MM-DiT returns a dict {shape, translation, rotation, scale}; SLAT/SS may
        # return a bare tensor. Guidance only touches the STAGE-1 SHAPE channel.
        if isinstance(v, dict):
            if "shape" in v:
                xt_shape = x_t["shape"] if isinstance(x_t, dict) else x_t
                if _is_ss_shape(xt_shape):
                    gv = guidance.velocity(xt_shape, t, fused_v=v["shape"])
                    v = dict(v); v["shape"] = v["shape"] + gv
            return v
        xt_shape = x_t["shape"] if isinstance(x_t, dict) else x_t
        if _is_ss_shape(xt_shape):
            return v + guidance.velocity(xt_shape, t, fused_v=v)
        return v
    return guided


In [ ]:
# 6.2 — Install the monkey-patch (composes with stock multi-diffusion)
# `sample_sparse_structure_multi_view` does a LOCAL import of inject_generator_multi_view,
# so overwriting the module attribute is picked up at call time with no source edits.
from contextlib import contextmanager
import sam3d_objects.pipeline.multi_view_utils as _mvu

ACTIVE_GUIDANCE = None          # set to a FootprintGuidance instance to enable
_ORIG_INJECT = getattr(_mvu, "_ORIG_INJECT", _mvu.inject_generator_multi_view)
_mvu._ORIG_INJECT = _ORIG_INJECT  # remember the genuine original (idempotent re-runs)

@contextmanager
def _guided_inject_generator_multi_view(generator, *args, **kwargs):
    with _ORIG_INJECT(generator, *args, **kwargs) as storage:
        if ACTIVE_GUIDANCE is not None:
            fused = generator._generate_dynamics            # the multi-diffusion fuser
            generator._generate_dynamics = make_guided_dynamics(fused, ACTIVE_GUIDANCE)
            try:
                yield storage
            finally:
                generator._generate_dynamics = fused
        else:
            yield storage

_mvu.inject_generator_multi_view = _guided_inject_generator_multi_view
print("Patched multi_view_utils.inject_generator_multi_view (guidance composes on top).")


## 7 — CPU smoke tests (no GPU / no model needed)

Validates the footprint → prior → guidance math with a tiny **stub decoder**, so
you can confirm shapes and that the guidance gradient is finite & shrinks the
energy — all on a Mac CPU before paying for a GPU run.


In [ ]:
# 7.0 — Logic tests with synthetic data and a stub ss_decoder
import torch, torch.nn as nn, numpy as np

# (a) footprint extraction on a synthetic square mask
synthetic_mask = np.zeros((400, 400), np.uint8)
synthetic_mask[120:300, 80:240] = 1
foot, poly = footprint_from_mask(synthetic_mask, out_size=128)
assert foot.shape == (128, 128) and foot.max() == 1.0
print("OK  footprint_from_mask ->", foot.shape, "occ", round(float(foot.mean()), 3))

# (b) extrusion
pr, fg = footprint_to_prior(foot, grid=16, up_axis="y")
assert pr.shape == (16, 16, 16)
print("OK  footprint_to_prior ->", pr.shape)

# (c) guidance with a differentiable stub decoder: (B,8,16,16,16) -> (B,1,16,16,16)
class StubDecoder(nn.Module):
    def forward(self, vol):                 # mean over the 8 channels, keepdim
        return vol.mean(dim=1, keepdim=True)

g = FootprintGuidance(fg, StubDecoder(), up_axis="y", scale=0.5, t_max=0.9,
                      mode="x_t", energy="bce", device="cpu")
z = torch.randn(1, 4096, 8, requires_grad=False)     # fake Stage-1 shape latent
v = g.velocity(z, torch.tensor(0.3))
assert v.shape == z.shape and torch.isfinite(v).all()
print("OK  guidance velocity ->", tuple(v.shape), "finite")

# (d) one guided step should not increase the energy
E0 = g._energy(z.float())
z2 = z + 0.5 * v                                       # emulate an Euler step
E1 = g._energy(z2.float())
print(f"OK  energy {float(E0):.4f} -> {float(E1):.4f} (expect <= , or close)")


## 8 — DA3 on the GROUND views only

The multi-diffusion still wants per-view pointmaps for the **ground** views
(Stage-1 physics CFG). We run DA3 on the ground images only. **Satellite
extrinsics are intentionally ignored** — that is the entire premise.

This reuses `scripts/run_da3.py` from MV-SAM3D (single pass). If you prefer the
chunked runner, swap the command.


In [ ]:
# 8.0 — Run DA3 on the ground images, then load da3_output.npz
import subprocess, numpy as np, glob, os

GROUND_IMAGES_DIR = GROUND_DIR / "images"
DA3_OUT = OUTPUT_DIR / "da3_ground"
DA3_OUT.mkdir(parents=True, exist_ok=True)
DA3_NPZ = DA3_OUT / "da3_output.npz"

# Resolve the DA3 weights to the actual snapshot dir. run_da3.py's auto-detect picks
# the HF cache ROOT (models--depth-anything--…), which from_pretrained can't load —
# we point --model_path at the snapshots/<hash>/ subdir (or fall back to the repo id).
DA3_MODEL_ID = "depth-anything/DA3NESTED-GIANT-LARGE"
_snap_glob = os.path.expanduser(
    "~/.cache/huggingface/hub/models--depth-anything--DA3NESTED-GIANT-LARGE/snapshots/*/")
_snaps = sorted(glob.glob(_snap_glob), key=os.path.getmtime, reverse=True)
DA3_MODEL_PATH = _snaps[0] if _snaps else globals().get("da3_cache", DA3_MODEL_ID)
print("DA3 model path:", DA3_MODEL_PATH)

if not DA3_NPZ.exists():
    cmd = [
        "python", str(MVSAM3D_ROOT / "scripts" / "run_da3.py"),
        "--image_dir", str(GROUND_IMAGES_DIR),
        "--output_dir", str(DA3_OUT),
        "--model_path", str(DA3_MODEL_PATH),
        "--process_res", "504",
        "--no_vis",
    ]
    print("Running:", " ".join(cmd))
    proc = subprocess.run(cmd, cwd=str(MVSAM3D_ROOT), capture_output=True, text=True)
    if proc.stdout:
        print(proc.stdout)
    if proc.returncode != 0:
        print("------ DA3 STDERR ------")
        print(proc.stderr)
        raise RuntimeError(
            f"run_da3.py failed (exit {proc.returncode}). See the DA3 STDERR above for the cause."
        )

da3 = np.load(DA3_NPZ)
da3_pointmaps = da3["pointmaps_sam3d"]      # (N, 3, H, W) — Stage-1 conditioning
da3_files = [Path(p).stem for p in da3["image_files"]]
print("DA3 pointmaps", da3_pointmaps.shape, "for stems", da3_files)


## 9 — Run guided multi-view reconstruction (ground views + footprint)

Loads the SAM3D/MV-SAM3D pipeline, matches DA3 pointmaps to ground views, builds
the `FootprintGuidance`, activates the patch, and calls `run_multi_view` with
**only the ground views**. The satellite footprint enters purely through the
guidance term inside Stage 1.

> Requires a GPU + the full MV-SAM3D environment (run the install cells of
> `mvsam3d_example_current.ipynb` first in the same session). On CPU this cell
> will not run — use Section 7 to validate the logic.


In [ ]:
# 9.0 — Build the pipeline (same checkpoint layout as mvsam3d_example_current.ipynb)
import os
from inference import Inference   # MV-SAM3D wrapper around the pipeline

# The working notebook exposes pipeline.yaml by symlinking the SAM 3D checkpoint into
# MV-SAM3D/checkpoints/hf, i.e. it runs (after snapshot_download of facebook/sam-3d-objects):
#   !ln -s /content/sam-3d-objects/checkpoints/hf/checkpoints /content/MV-SAM3D/checkpoints/hf
# So run the heavy install + checkpoint download from mvsam3d_example_current.ipynb first.
MODEL_TAG = "hf"
PIPELINE_YAML = MVSAM3D_ROOT / "checkpoints" / MODEL_TAG / "pipeline.yaml"

if not PIPELINE_YAML.exists():
    # Recreate the same symlink if the SAM 3D checkpoint is already downloaded
    src = SAM3D_ROOT / "checkpoints" / "hf" / "checkpoints"
    dst = MVSAM3D_ROOT / "checkpoints" / "hf"
    if src.exists() and not dst.exists():
        dst.parent.mkdir(parents=True, exist_ok=True)
        os.symlink(src, dst)
        print("Created checkpoint symlink:", dst, "->", src)

assert PIPELINE_YAML.exists(), (
    f"{PIPELINE_YAML} missing. Download the SAM 3D checkpoint and create the "
    f"checkpoints/hf symlink via mvsam3d_example_current.ipynb first."
)

inference = Inference(str(PIPELINE_YAML))
pipeline = inference._pipeline
ss_decoder = pipeline.models["ss_decoder"]      # <- handle used by the guidance
print("Pipeline loaded. ss_decoder:", type(ss_decoder).__name__)


In [ ]:
# 9.1 — Match DA3 pointmaps to ground views (by filename stem)
view_pointmaps = []
for name in ground_names:
    if name in da3_files:
        view_pointmaps.append(da3_pointmaps[da3_files.index(name)])
    else:
        print(f"  WARN: no DA3 pointmap for ground view {name}; using None")
        view_pointmaps.append(None)
print(f"Matched {sum(p is not None for p in view_pointmaps)}/{len(ground_names)} pointmaps")


In [ ]:
# 9.1b — Auto-register the OSM footprint to the model's canonical frame (RUN ONCE before 9.2)
# The model's Stage-1 frame is object-centric: arbitrary in-plane rotation + position, and it
# has no idea where 'north' is. The OSM footprint is north-up. UP_AXIS/flip knobs only express
# 90-degree flips, NOT the ~40-degree rotation + translation seen in the overlay, so the raw
# silhouette guidance pulls occupancy toward a misplaced target and fights the image prior
# (low IoU -> smeared L). Here we (1) run Stage-1 with guidance OFF to get the model's natural
# silhouette M, then (2) fit a similarity transform (flip + rotation + scale + translation) via
# image moments + a 4-way IoU search, and overwrite foot_grid with the ALIGNED target.
import numpy as np, torch, cv2
import matplotlib.pyplot as plt

_GA_BACKUP = globals().get("ACTIVE_GUIDANCE", None)
ACTIVE_GUIDANCE = None   # baseline MUST be unguided
print("Running unguided Stage-1 baseline for registration (a few min) ...")
_ss0 = pipeline.run_multi_view(
    view_images=ground_images, view_masks=ground_masks, view_pointmaps=view_pointmaps,
    mode="multidiffusion", seed=0, stage1_inference_steps=25,
    ss_weighting=True, ss_entropy_alpha=60.0, ss_warmup_steps=1, stage1_only=True,
)
_c = _ss0["coords"].detach().cpu().numpy()
_occ0 = np.zeros((64, 64, 64), bool); _occ0[_c[:, 1], _c[:, 2], _c[:, 3]] = True
_axis = {"x": 0, "y": 1, "z": 2}[UP_AXIS]
M = _occ0.max(axis=_axis).astype(np.float32)          # model silhouette in the guidance plane
G = M.shape[0]

T = torch.as_tensor(np.asarray(foot_grid), dtype=torch.float32)[None, None]
T = torch.nn.functional.interpolate(T, size=(G, G), mode="bilinear", align_corners=False)[0, 0].numpy()

def _moments(m):
    ys, xs = np.nonzero(m > 0.5)
    if len(xs) < 5: return None
    cx, cy = xs.mean(), ys.mean()
    x, y = xs - cx, ys - cy
    u20, u02, u11 = (x * x).mean(), (y * y).mean(), (x * y).mean()
    return np.array([cx, cy]), 0.5 * np.arctan2(2 * u11, u20 - u02), float((m > 0.5).sum())

def _iou(a, b):
    a, b = a > 0.5, b > 0.5; u = (a | b).sum()
    return float((a & b).sum() / u) if u else 0.0

_mM = _moments(M)
assert _mM is not None, "Baseline occupancy is empty - cannot register."
cM, thM, areaM = _mM

# Per-flip base + its centroid/orientation/area (computed once)
_cands = {}
for flip in (False, True):
    base = T[:, ::-1] if flip else T
    mb = _moments(base)
    if mb is not None: _cands[flip] = (base, mb[0], mb[1], mb[2])   # base, centroid, theta, area

def _warp(flip, ang, scale, dx, dy):
    base, cT, _, _ = _cands[flip]
    R = cv2.getRotationMatrix2D((float(cT[0]), float(cT[1])), float(ang), float(scale))
    R[0, 2] += (cM[0] - cT[0]) + dx
    R[1, 2] += (cM[1] - cT[1]) + dy
    return cv2.warpAffine(base.astype(np.float32), R, (G, G), flags=cv2.INTER_LINEAR)

# (a) moment SEED: align principal axis + area, resolving the flip / 180-degree ambiguity by IoU
best = None   # (iou, flip, ang, scale, dx, dy)
for flip, (base, cT, thT, areaT) in _cands.items():
    scale0 = float(np.sqrt(areaM / max(areaT, 1.0)))
    for extra in (0.0, np.pi):
        ang0 = float(np.degrees(thM - thT + extra))
        sc = _iou(_warp(flip, ang0, scale0, 0.0, 0.0), M)
        if best is None or sc > best[0]: best = (sc, flip, ang0, scale0, 0.0, 0.0)
print(f"seed IoU {best[0]:.3f}  (flip={best[1]} ang={best[2]:.1f} scale={best[3]:.2f})")

# (b) coarse-to-fine grid search around the seed: (angle, scale) -> (shift) -> (fine joint)
_flip = best[1]
def _grid(c, rng, step): return c + np.arange(-rng, rng + 1e-9, step) if rng else np.array([c])
for (a_rng, a_step, s_rng, s_step, d_rng, d_step) in [
        (20.0, 2.0, 0.30, 0.05, 0.0, 1.0),    # angle + scale (no shift yet)
        (0.0,  1.0, 0.0,  1.0,  8.0, 1.0),    # translation only
        (4.0,  1.0, 0.06, 0.02, 2.0, 1.0)]:   # fine joint refine
    _, _, ba, bs, bdx, bdy = best
    angs   = _grid(ba,  a_rng, a_step)
    scales = bs * (1.0 + _grid(0.0, s_rng, s_step)) if s_rng else np.array([bs])
    dxs    = _grid(bdx, d_rng, d_step)
    dys    = _grid(bdy, d_rng, d_step)
    for ang in angs:
        for scale in scales:
            for dx in dxs:
                for dy in dys:
                    sc = _iou(_warp(_flip, ang, scale, dx, dy), M)
                    if sc > best[0]: best = (sc, _flip, float(ang), float(scale), float(dx), float(dy))

warp = _warp(best[1], best[2], best[3], best[4], best[5])
iou0 = _iou(T, M)
print(f"Registration IoU: {iou0:.3f} (raw) -> {best[0]:.3f} (aligned) | "
      f"flip={best[1]} angle={best[2]:.1f} scale={best[3]:.2f} shift=({best[4]:.0f},{best[5]:.0f})")

foot_grid_raw = np.asarray(foot_grid).copy()
foot_grid = np.clip(warp, 0.0, 1.0)                    # cell 9.2 will guide with this ALIGNED target
ACTIVE_GUIDANCE = _GA_BACKUP

fig, ax = plt.subplots(1, 4, figsize=(16, 4))
ax[0].imshow(M, origin="lower", cmap="Blues");          ax[0].set_title("Unguided Stage-1 (M)")
ax[1].imshow(T, origin="lower", cmap="Greens");         ax[1].set_title(f"OSM raw (IoU {iou0:.2f})")
ax[2].imshow(foot_grid, origin="lower", cmap="Greens"); ax[2].set_title(f"OSM aligned (IoU {best[0]:.2f})")
ov = np.zeros((G, G, 3)); ov[..., 1] = foot_grid > 0.5; ov[..., 2] = M > 0.5
ax[3].imshow(ov, origin="lower");                       ax[3].set_title("overlay  G=OSM  B=model")
for a in ax: a.set_xticks([]); a.set_yticks([])
plt.tight_layout(); plt.show()
print("foot_grid replaced with the ALIGNED target. Now run cell 9.2 to guide with it.")
print("If alignment still looks off, the baseline shape is ambiguous - bump GUIDANCE_SCALE/TMAX after.")


In [ ]:
# 9.2 — Activate footprint guidance and run
import torch

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# Build guidance with the REAL ss_decoder at full resolution
ACTIVE_GUIDANCE = FootprintGuidance(
    foot_grid, ss_decoder,
    up_axis=UP_AXIS, scale=GUIDANCE_SCALE, t_max=GUIDANCE_TMAX,
    mode=GUIDANCE_MODE, energy=GUIDANCE_ENERGY, device=DEVICE,
)

try:
    result = pipeline.run_multi_view(
        view_images=ground_images,
        view_masks=ground_masks,
        view_pointmaps=view_pointmaps,     # DA3 pointmaps for GROUND views only
        mode="multidiffusion",
        seed=0,
        stage1_inference_steps=25,
        stage2_inference_steps=25,
        decode_formats=["mesh", "gaussian"],
        with_mesh_postprocess=False,
        with_texture_baking=False,
        use_vertex_color=True,
        # Stage-1 multi-diffusion uses our patched fuser -> guidance is injected.
        ss_weighting=True, ss_entropy_alpha=60.0, ss_warmup_steps=1,
        # Stage-2 stays stock (footprint does not constrain SLAT features).
        weighting_config=None,
    )
finally:
    ACTIVE_GUIDANCE = None   # always disable the guidance after the run

print("Done. Result keys:", list(result.keys()))


In [ ]:
# 9.3 — Save outputs to Drive
import shutil
from datetime import datetime

run_dir = OUTPUT_DIR / ("guided_" + datetime.now().strftime("%Y%m%d_%H%M%S"))
run_dir.mkdir(parents=True, exist_ok=True)

# Decoded assets (keys depend on decode_formats / pipeline version)
for key in ("mesh", "gaussian", "glb"):
    obj = result.get(key)
    if obj is None:
        continue
    try:
        if hasattr(obj, "export"):                       # trimesh-like
            obj.export(str(run_dir / f"result_{key}.glb"))
        elif hasattr(obj, "save_ply"):
            obj.save_ply(str(run_dir / f"result_{key}.ply"))
    except Exception as e:
        print(f"  could not export {key}: {e}")

# Always keep a record of the footprint + prior used
np.savez(run_dir / "footprint_prior.npz", footprint=footprint, foot_grid=foot_grid, prior3d=prior3d)
print("Saved to", run_dir)


## 9.5 — Inspect the Stage-1 rough geometry (debug the footprint guidance)

Stage 1 produces a coarse **occupancy** on a 64³ grid (the `coords` voxels). This is
exactly what the OSM footprint guidance steered. We reconstruct that voxel grid and
look at its **top-down silhouette vs the OSM target** to see whether the footprint
actually took (U vs L), and whether the mismatch is an *orientation* problem
(`UP_AXIS` / `FOOTPRINT_FLIP_*`) or a *guidance-strength* problem.

If the U is visible in a **different** projection than the one guidance targeted, the
`UP_AXIS`/flip calibration is wrong. If no projection shows a U, the guidance was too
weak (raise `GUIDANCE_SCALE` / `GUIDANCE_TMAX`) or is fighting the ground-view CFG.


In [ ]:
# 9.5a — Get the Stage-1 occupancy voxels (reuse the full run, else run stage1_only)
import numpy as np, torch

def _coords_to_occ(coords, G=64):
    c = coords.detach().cpu().numpy() if torch.is_tensor(coords) else np.asarray(coords)
    occ = np.zeros((G, G, G), dtype=bool)
    occ[c[:, 1], c[:, 2], c[:, 3]] = True      # coords are [batch, x, y, z] in the 64^3 grid
    return occ

ss = None
if "result" in globals() and isinstance(result, dict) and "coords" in result:
    ss = result
    print("Reusing coords from the previous full run (result['coords']).")
else:
    print("Running Stage-1 only (guided) to obtain the occupancy...")
    ACTIVE_GUIDANCE = FootprintGuidance(
        foot_grid, ss_decoder, up_axis=UP_AXIS, scale=GUIDANCE_SCALE, t_max=GUIDANCE_TMAX,
        mode=GUIDANCE_MODE, energy=GUIDANCE_ENERGY, device=DEVICE,
    )
    try:
        ss = pipeline.run_multi_view(
            view_images=ground_images, view_masks=ground_masks, view_pointmaps=view_pointmaps,
            mode="multidiffusion", seed=0, stage1_inference_steps=25,
            ss_weighting=True, ss_entropy_alpha=60.0, ss_warmup_steps=1,
            stage1_only=True,
        )
    finally:
        ACTIVE_GUIDANCE = None

occ = _coords_to_occ(ss["coords"])
print("Occupied voxels:", int(occ.sum()), "/", occ.size)


In [ ]:
# 9.5b — Compare the Stage-1 footprint to the OSM target + 3 orthographic projections
import numpy as np, matplotlib.pyplot as plt
import torch.nn.functional as F
import torch

# occ is indexed [x, y, z]. Project the SAME way the guidance does (max over UP_AXIS).
proj = {
    "x": occ.max(axis=0),   # -> (Y, Z)
    "y": occ.max(axis=1),   # -> (X, Z)   <- footprint plane when UP_AXIS='y'
    "z": occ.max(axis=2),   # -> (X, Y)
}
G = occ.shape[0]

# OSM target resized to the grid for overlay
tgt = torch.as_tensor(np.asarray(foot_grid), dtype=torch.float32)[None, None]
tgt = F.interpolate(tgt, size=(G, G), mode="bilinear", align_corners=False)[0, 0].numpy()
tgt_bin = tgt > 0.5

fp = proj[UP_AXIS].astype(bool)            # the projection the guidance actually targeted
inter = (fp & tgt_bin).sum(); union = (fp | tgt_bin).sum()
iou = inter / union if union else 0.0

fig, ax = plt.subplots(1, 5, figsize=(22, 4.5))
ax[0].imshow(tgt_bin, cmap="Greens", origin="lower");      ax[0].set_title("OSM target footprint")
ax[1].imshow(fp, cmap="Blues", origin="lower");            ax[1].set_title(f"Stage-1 footprint (up={UP_AXIS})")
ov = np.zeros((*fp.shape, 3)); ov[..., 1] = tgt_bin; ov[..., 2] = fp
ax[2].imshow(ov, origin="lower");                          ax[2].set_title(f"overlay  (IoU={iou:.2f})\nGreen=OSM  Blue=Stage1")
ax[3].imshow(proj["x"], cmap="gray", origin="lower");      ax[3].set_title("proj over X  (Y,Z)")
ax[4].imshow(proj["z"], cmap="gray", origin="lower");      ax[4].set_title("proj over Z  (X,Y)")
for a in ax: a.set_xticks([]); a.set_yticks([])
plt.tight_layout(); plt.show()
print(f"Footprint IoU (Stage-1 vs OSM, up={UP_AXIS}): {iou:.3f}")
print("If a U-shape shows up in 'proj over X' or 'proj over Z' instead, set UP_AXIS to that axis.")

# 3D voxel view (subsampled for speed)
xs, ys, zs = np.where(occ)
if len(xs) > 8000:
    sel = np.random.choice(len(xs), 8000, replace=False); xs, ys, zs = xs[sel], ys[sel], zs[sel]
fig = plt.figure(figsize=(6, 6)); ax3 = fig.add_subplot(111, projection="3d")
ax3.scatter(xs, ys, zs, c=ys, s=2, cmap="viridis")
ax3.set_xlabel("X"); ax3.set_ylabel("Y"); ax3.set_zlabel("Z"); ax3.set_title("Stage-1 occupancy (3D)")
plt.tight_layout(); plt.show()

# Save the figure next to the run outputs if a run_dir exists
try:
    out_png = (run_dir if "run_dir" in globals() else OUTPUT_DIR) / "stage1_footprint_debug.png"
    fig.savefig(out_png, dpi=120, bbox_inches="tight"); print("Saved 3D view to", out_png)
except Exception as e:
    print("(figure not saved:", e, ")")


In [ ]:
# 9.5c — Convert the Stage-1 occupancy into a real coarse MESH (marching cubes)
#
# Stage 1 only decides occupancy. There is no texture/appearance yet (that is Stage 2),
# but we can still extract a watertight coarse surface to inspect the U-vs-L geometry in
# a normal 3D viewer. We decode the dense shape latent the SAME way the guidance does
# (B,8,16,16,16) -> (B,1,64,64,64) logits, then marching-cubes the probability at 0.5.
import numpy as np, torch
try:
    from skimage import measure
except ModuleNotFoundError:
    import subprocess, sys
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "scikit-image"], check=True)
    from skimage import measure
import trimesh

# 1) Continuous 64^3 occupancy from the decoder (smoother than the thresholded voxels).
prob = None
shape_latent = ss.get("shape") if isinstance(ss, dict) else None
if shape_latent is not None and torch.is_tensor(shape_latent) and shape_latent.dim() == 3:
    with torch.no_grad(), torch.autocast(device_type=str(DEVICE).split(":")[0], enabled=True):
        B = shape_latent.shape[0]
        vol = shape_latent.permute(0, 2, 1).contiguous().view(B, 8, 16, 16, 16)
        logits = ss_decoder(vol.to(DEVICE))                 # (B,1,64,64,64)
    prob = torch.sigmoid(logits[0, 0].float()).cpu().numpy()
    print("Decoded continuous occupancy:", prob.shape, "max", round(float(prob.max()), 3))
else:
    print("No 'shape' latent available -> using the thresholded voxel grid instead.")

# 2) Marching cubes. Fall back to the binary voxel grid if the soft volume is degenerate.
def _mc(vol3d, level):
    v, f, n, _ = measure.marching_cubes(vol3d, level=level)
    return v, f, n

try:
    verts, faces, normals = _mc(prob, 0.5)
    src = "decoder occupancy @0.5"
except Exception as e:
    print("(soft marching cubes failed:", e, "-> using binary voxels)")
    verts, faces, normals = _mc(occ.astype(np.float32), 0.5)
    src = "binary voxels @0.5"

# Center + scale verts into the canonical [-0.5, 0.5] cube (grid is 64^3)
verts = verts / 64.0 - 0.5
mesh = trimesh.Trimesh(vertices=verts, faces=faces, vertex_normals=normals, process=True)
print(f"Stage-1 coarse mesh ({src}): {len(mesh.vertices)} verts, {len(mesh.faces)} faces")

# 3) Save for download / external viewer (both OBJ and GLB)
out_dir = run_dir if "run_dir" in globals() else OUTPUT_DIR
obj_path = out_dir / "stage1_coarse_mesh.obj"
glb_path = out_dir / "stage1_coarse_mesh.glb"
mesh.export(obj_path); mesh.export(glb_path)
print("Saved:", obj_path, "and", glb_path)

# 4) Quick inline preview from a few angles (matplotlib trisurf)
import matplotlib.pyplot as plt
vx, vy, vz = mesh.vertices[:, 0], mesh.vertices[:, 1], mesh.vertices[:, 2]
fig = plt.figure(figsize=(15, 5))
for i, (az, el) in enumerate([(-60, 25), (30, 25), (90, 80)]):  # 3rd view is near top-down
    ax = fig.add_subplot(1, 3, i + 1, projection="3d")
    ax.plot_trisurf(vx, vy, vz, triangles=mesh.faces, cmap="viridis", linewidth=0, antialiased=False)
    ax.view_init(elev=el, azim=az); ax.set_box_aspect((1, 1, 1))
    ax.set_xlabel("X"); ax.set_ylabel("Y"); ax.set_zlabel("Z")
    ax.set_title(f"Stage-1 coarse mesh ({az},{el})")
plt.tight_layout(); plt.show()
print("Tip: the top-down-ish view (rightmost) should show the U footprint if guidance worked.")


In [ ]:
# 9.6 — Copy the run to Google Drive (persists after the Colab session ends)
# `run_dir` above lives in local Colab storage (/content/flowmatch_out) which is wiped
# when the runtime stops. This mirrors the whole run folder onto your Drive, including the
# Stage-1 debug outputs from 9.5 (stage1_footprint_debug.png, stage1_coarse_mesh.obj/.glb).
import shutil
from pathlib import Path

# Default: a results folder on Drive, namespaced by the scene folder name. EDIT if you like.
DRIVE_RESULTS = Path("/content/drive/MyDrive/flowmatch_results") / DATA_ROOT.name
DRIVE_RESULTS.mkdir(parents=True, exist_ok=True)

drive_run_dir = DRIVE_RESULTS / run_dir.name
if drive_run_dir.exists():
    shutil.rmtree(drive_run_dir)
shutil.copytree(run_dir, drive_run_dir)

print("Copied run to Drive:", drive_run_dir)
for f in sorted(drive_run_dir.iterdir()):
    print("  ", f.name, f"({f.stat().st_size/1e6:.2f} MB)")


## 10 — Notes, limitations & next steps

**What this prototype does**
- Ground views run stock multi-diffusion (their DINOv2 features condition Stage 1 & 2).
- The satellite contributes **only** a top-down footprint, injected as a velocity
  guidance term on the Stage-1 occupancy flow. Satellite extrinsics are unused.

**Known approximations (research TODOs)**
1. **Canonical alignment.** Silhouette guidance assumes the footprint top-down
   plane matches the canonical `UP_AXIS` plane. Rotation in the ground plane,
   anisotropic scale, and translation are *not* solved here. Use the Stage-1
   occupancy overlay to calibrate `UP_AXIS` / flips; for full alignment, estimate
   the in-plane rotation (e.g. align footprint principal axes to the decoded
   silhouette) before extrusion.
2. **Guidance ↔ physics-CFG interaction.** Stage 1 already runs `PointmapCFG`
   (3-way CFG with DA3 pointmaps). Our term adds to the *fused* velocity after
   that. If they fight, lower `GUIDANCE_SCALE` or `GUIDANCE_TMAX`.
3. **Flow convention.** `mode="x1"` assumes a non-reversed `t:0→1` schedule
   (`reversed_timestamp=False`). `mode="x_t"` is convention-free; prefer it unless
   you verify the schedule.
4. **Silhouette vs volume.** We match a top-down silhouette. A full volumetric
   prior (the `prior3d` we built) needs the Stage-1 object pose to align — a
   natural next experiment via a second guidance pass after pose is known.

**Stronger future design (token injection instead of guidance)**
Replace the guidance with a learned **geometry adapter** that turns the footprint
into condition tokens shaped like DINO tokens `(B, L, D)`, and swap them in for
satellite views inside `get_multi_view_condition_input`
(`inference_pipeline.py:964`). That keeps the satellite as a real "view" but with
*geometry* tokens instead of image tokens. It needs training; the guidance route
here needs none, which is why it is the default.
